# Automated StoryMap Creator

In [1]:
import re
from bs4 import BeautifulSoup
from arcgis.gis import GIS
from IPython.display import Markdown, display
from arcgis.geocoding import geocode
import time
from arcgis.geocoding import get_geocoders, batch_geocode
from arcgis.apps.storymap import StoryMap, story, story_content
from arcgis.map import Map
from arcgis.apps.storymap.story import StoryMap
from arcgis.apps.storymap.story_content import Image, TextStyles, Video, Audio, Embed, Map, Text, Button, Gallery, Swipe, Sidecar, Timeline
gis = GIS("home")

## Determining the location

In [2]:
web_map_list = []

while True:
    web_map_prompt = input("Input one or more AGOL Web Map item IDs (separated by commas or spaces):").strip()

    # Split by commas or spaces
    item_ids = [id_str.strip() for id_str in web_map_prompt.replace(",", " ").split() if id_str.strip()]

    if not item_ids:
        print("No item IDs entered. Please try again.")
        continue

    valid_count = 0

    for item_id in item_ids:
        web_map = gis.content.get(item_id)

        if web_map is None:
            print(f"No item found with ID: {item_id}")
            continue

        if web_map.type != 'Web Map':
            print(f"Item {item_id} is of type '{web_map.type}', not 'Web Map'.")
            continue

        web_map_list.append(web_map)
        print(f"Web Map found: {web_map.title}")
        valid_count += 1

    if valid_count > 0:
        print(f"\n{valid_count} valid Web Map(s) added to the list. Proceeding...")
        break

# Error-catching input logic to obtain a location
while True:
    location_prompt = input("Enter a city of interest: ")
    result = geocode(location_prompt)

    if len(result) < 1:
        print('No geocoding result found for the provided city, please ensure your spelling is correct.')
        continue

    city_location = result[0]['location']
    city_name = result[0]['address']

    break

Input one or more AGOL Web Map item IDs (separated by commas or spaces): 7b67417ceb5249cbb5fc904469d5d716


Web Map found: Average Wildfire Hazard Potential in the US

1 valid Web Map(s) added to the list. Proceeding...


Enter a city of interest:  Hays, KS


In [3]:
for web_map_item in web_map_list:

    print(f"Creating StoryMap for {web_map.title} centered on {city_name}")


    print(f"First, generating the Web Map")
       
    # Create map from web map
    demo_map = gis.map(web_map_item)

    # center it on the location
    demo_map.extent = {
        "xmin": city_location['x'] - 1,
        "ymin": city_location['y'] - 1,
        "xmax": city_location['x'] + 1,
        "ymax": city_location['y'] + 1,
        "spatialReference": {"wkid": 4326}
    }

    # create storymap object 
    new_story = StoryMap()
    
    # populate w/ info from webmap
    # Gathering title
    current_title = f"{web_map_item.title} centered on {city_name}"
    title_prompt = input(f"The title for the StoryMap will default to: \n\n'{current_title}'\n\nWould you like to use this as the StoryMap title? If so, type a new title in the dialog box. If not, just hit Enter.")
    if title_prompt:
        current_title = title_prompt  
    
    # adjusting the title and summary
    new_story.content_list[0].title = current_title
    new_story.content_list[0].summary = "This StoryMap has been created using the ArcGIS API for Python."
    
    preexisting_storymap = gis.content.search(query=f"title:{current_title}", item_type="StoryMap")
    if len(preexisting_storymap) > 0:
        print(f"StoryMap already found with the title '{current_title}', deleting it...")
        preexisting_storymap[0].delete()
        time.sleep(20) # adding a sleep to allow AGOL to refresh 
    
    # Gathering description
    current_description = web_map_item.description
    current_description_cleaned = BeautifulSoup(current_description).get_text().replace("\xa0", "")
    description_prompt = input(f"\nThe description for the StoryMap will default to: \n\n'{current_description_cleaned}'\n\nWould you like to use this as the StoryMap description? If so, type paste a new description in the dialog box. If not, just hit Enter.")
    if description_prompt:
        current_description = description_prompt
    
    print(f"-"*90)
    print("Extracting information from the Web Map to populate the StoryMap...")

    # setting the Web Map's thumbnail as the StoryMap logo
    print("Adding an image...")
    thumbnail_path = web_map_item.download_thumbnail()
    time.sleep(20)
    image = Image(thumbnail_path, display='full', full_view=True)
    new_story.add(image, caption=f"image of {current_title}", alt_text=f"image of {current_title}", position=2)
    new_story.add() # empty node for spacing
    
    # adding a map, using the title of the Web Map as the alt-text
    print("Adding the Web Map...")
    web_map_item_map = Map(web_map_item)
    new_story.add(web_map_item_map, caption=f"Map of {current_title}", alt_text=f"Map of {current_title}")
    web_map_item_map.set_viewpoint(extent={
        "xmin": city_location['x'] - 1,
        "ymin": city_location['y'] - 1,
        "xmax": city_location['x'] + 1,
        "ymax": city_location['y'] + 1,
        "spatialReference": {"wkid": 4326}
    })
    new_story.add() # empty node for spacing
    
    # adjusting the credits
    new_story.credits("StoryMaps" , "Python API", "Thank You for Reading", "Esri Living Atlas of the World")

    # saving the StoryMap we made
    print("Saving the StoryMap...")
    new_story.save(title=current_title, tags=['python', 'automation', 'StoryMap', 'API'])

    
    # update storymap item details
    print("Waiting 10 seconds before searching for StoryMap item.")
    time.sleep(20)
    
    # then updating the StoryMap using the actual AGOL item
    # first we have to grab it from our content
    story_item_search = gis.content.search(query=f"title:{current_title}", item_type="StoryMap")
    
    if len(story_item_search) > 0:
        story_item = story_item_search[0]
    
        print("Waiting 10 seconds before updating StoryMap's description.")
        time.sleep(20)
        
        # then change the description
        story_item.update(
            item_properties={
                "access": "public",
                "description": str(current_description),
                "snippet": "This StoryMap was made with the ArcGIS API for Python",
            })
        
        print("Waiting 10 seconds before updating StoryMap's thumbnail.")
        time.sleep(20)
        
        # then update the thumbnail
        story_item.update_thumbnail(thumbnail_path)
    
        print("Done updating the StoryMap.\n")

Creating StoryMap for Average Wildfire Hazard Potential in the US centered on Hays, Kansas
First, generating the Web Map


The title for the StoryMap will default to: 

'Average Wildfire Hazard Potential in the US centered on Hays, Kansas'

Would you like to use this as the StoryMap title? If so, type a new title in the dialog box. If not, just hit Enter. 

The description for the StoryMap will default to: 

'This web map shows the wildfire hazard potential (WHP) for the conterminous United States aggregated from states to block groups and 50 km hex bins. The data is from theUSDA Forest Service Fire Modeling Instituteproviding an index of WHP at a 270 meter resolution. Wildfire hazard potentialprovides information on the relative potential for wildfire that would be difficult for fire crews to contain. Areas with higher wildfire potential values represent fuels with a higher likelihood of experiencing high-intensity fire with torching, crowning, and other forms of extreme fire behavior. A score of 5 is very high risk and a score between 0-1 is non-burnable area such as water or asphalt.On its own, WHP is n

------------------------------------------------------------------------------------------
Extracting information from the Web Map to populate the StoryMap...
Adding an image...
Adding the Web Map...
Saving the StoryMap...
Waiting 10 seconds before searching for StoryMap item.
Waiting 10 seconds before updating StoryMap's description.
Waiting 10 seconds before updating StoryMap's thumbnail.
Done updating the StoryMap.



*Examples*\
\
**USA Current Wildfires:** 89ff30d783b849c8b22fc812d4c2f205\
**Wind Farms and Live Weather Assessment:** 6109d505d1d044f689bdafcb35153f8e\
**Projected Population Growth 2023-2028:** de22c18ec9d945e9b222c81707af2c4d\
\
**Altogether**: 89ff30d783b849c8b22fc812d4c2f205 6109d505d1d044f689bdafcb35153f8e de22c18ec9d945e9b222c81707af2c4d